In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
# page 135
from langchain_community.document_loaders import PDFPlumberLoader
import pdfplumber

loader = PDFPlumberLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
docs = docs[3:-1]
len(docs)

19

In [8]:
# page 136
docs[0].metadata

{'source': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'page': 3,
 'total_pages': 23,
 'Author': 'dj',
 'Creator': 'Hwp 2018 10.0.0.13462',
 'Producer': 'Hancom PDF 1.3.0.542',
 'CreationDate': "D:20231208132838+09'00'",
 'ModDate': "D:20231208132838+09'00'",
 'PDFVersion': '1.4'}

In [9]:
for doc in docs:
    doc.metadata["filename"] = doc.metadata["source"]

In [10]:
from ragas.testset import TestsetGenerator

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 데이터셋 생성 모델
generator_llm = ChatOpenAI(model="gpt-4o-mini")
# 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Ragas 0.4.x에서는 KeyphraseExtractor와 critic_llm을 직접 구성하지 않습니다.
generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=embeddings,
)

In [ ]:
# page 138
from ragas.testset.transforms                       import SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors            import NERExtractor
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

transforms = [                                                      # 페이지를 분할하지 않고 각 DOCUMENT에 필요한 정보를 추출
    
    SummaryExtractor(llm=generator.llm),                            # LLM으로 문서 내용을 요약

    EmbeddingExtractor(                                             # 지정한 텍스트 속성을 숫자 벡터로 변환
        embedding_model    =generator.embedding_model,
        property_name      ="summary_embedding",
        embed_property_name="summary",
    ),
    NERExtractor(llm=generator.llm),                                # 개체명 인식(Named Entity Recognition)으로 사람·기관·장소 등 주요 개체를 추출
]

query_distribution = [                                              # 한 페이지의 개체 정보를 바탕으로 질문 생성
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm), 1.0),    # 하나의 문맥에서 답할 수 있는 구체적인 질문과 참조 정답을 생성
]

In [ ]:
# 테스트셋 생성됨.
testset = generator.generate_with_langchain_docs(
    documents           =docs,                  # PDF 문서에서 추출한 페이지를 DOCUMENT로 변환하여 전달

    testset_size        =10,                    # 생성할 테스트셋의 질문 수

    transforms          =transforms,            # 각 DOCUMENT에 적용할 변환기(요약, 임베딩, 개체명 인식)

    query_distribution  =query_distribution,    # 한 페이지의 개체 정보를 바탕으로 질문 생성
    
    raise_exceptions    =True,                  # 질문 생성 중 오류 발생 시 예외를 발생시키도록 설정
)

Applying SummaryExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
# page 139
test_df = testset.to_pandas()

if test_df.empty:
    raise RuntimeError(
        "샘플이 0개입니다. 개체 추출 결과와 페르소나 매칭을 확인하세요."
    )

print(f"생성된 샘플 수: {len(test_df)}")
#test_df.head()
# print(test_df)
test_df

생성된 샘플 수: 10


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What are the key contents of the executive ord...,"[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하고...",The executive order signed by President Biden ...,AI Safety Researcher,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,영국은 G7의 AI 국제 행동강령에 어떤 역할을 하고 있나요?,"[SPRi AI Brief |\n2023-12월호\nG7, 히로시마 AI 프로세스를...","영국은 G7의 주요 7개국 중 하나로, 2023년 10월 30일 '히로시마 AI 프...",AI Safety Researcher,MISSPELLED,LONG,single_hop_specific_query_synthesizer
2,한국은 AI 안전성 정상회의에서 뭐 했어요?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n영국 AI 안전...,한국은 영국 정부와 6개월 뒤에 온라인으로 AI 미니 정상회의를 공동 개최하기로 합...,AI Safety Researcher,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
3,미국 법원은 뭐 했어요?,"[SPRi AI Brief |\n2023-12월호\n미국 법원, 예술가들이 생성 A...","미국 캘리포니아 북부지방법원은 예술가들이 미드저니, 스태빌리티AI, 디비언트아트를 ...",AI Safety Researcher,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
4,What concerns does the FTC have regarding the ...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 연방거래위...,The FTC has raised concerns that the AI assist...,AI Safety Researcher,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
5,What role does Mistral play in the EU AI regul...,"[SPRi AI Brief |\n2023-12월호\nEU AI 법 3자 협상, 기반...","Mistral, a French AI company, is leading the o...",AI Safety Researcher,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
6,마이크로소프트는 AI 안전 기금 조성에 어떤 역할을 하고 있나요?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 프런티어 ...,"마이크로소프트는 구글, 앤스로픽, 오픈AI와 함께 프런티어 모델 포럼에 참여하여 A...",AI Safety Researcher,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
7,What is the purpose of the 데이터 출처 탐색기 platform?,"[SPRi AI Brief |\n2023-12월호\n코히어, 데이터 투명성 확보를 ...",The 데이터 출처 탐색기 platform aims to enhance data t...,AI Safety Researcher,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
8,통이치엔원 2.0 뭐가 좋고 어떻게 쓰는거야?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n알리바바 클라우...,"통이치엔원 2.0은 알리바바 클라우드가 공개한 최신 LLM으로, 복잡한 지침 이해,...",AI Policy Analyst,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
9,What is the significance of 라마(Llama)2 in the ...,"[SPRi AI Brief |\n2023-12월호\n삼성전자, 자체 개발 생성 AI...",라마(Llama)2 is expected to compete with 삼성전자의 삼...,AI Safety Researcher,MISSPELLED,SHORT,single_hop_specific_query_synthesizer


In [15]:
test_df.to_csv("../data/ragas_synthetic_dataset.csv", index=False, encoding="utf-8-sig")
print("저장완료")

저장완료
